In [ ]:
# ============================================================
# PRACTICAL: RECURRENT NEURAL NETWORK (RNN)
# ============================================================

# Install the required libraries if needed
# !pip install tensorflow yfinance scikit-learn matplotlib


# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

# Import NumPy for numerical calculations
import numpy as np

# Import Pandas for data handling
import pandas as pd

# Import Matplotlib for plotting graphs
import matplotlib.pyplot as plt

# Import yfinance to download Google stock prices
import yfinance as yf

# Import MinMaxScaler for normalizing data
from sklearn.preprocessing import MinMaxScaler

# Import TensorFlow
import tensorflow as tf

# Import Sequential model
from tensorflow.keras.models import Sequential

# Import RNN, CNN, and other neural network layers
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
# GOOGLE STOCK PRICE PREDICTION USING RNN
# ============================================================

print("\n==========================================")
print("SUBQUESTION 1: GOOGLE STOCK PRICE RNN")
print("==========================================")


# Download Google stock price data
data = yf.download(
    "GOOGL",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=False
)


# Select the closing price
prices = data["Close"]


# Handle the possibility of a multi-level column
if isinstance(prices, pd.DataFrame):

    # Select the GOOGL column if available
    if "GOOGL" in prices.columns:
        prices = prices["GOOGL"]

    # Otherwise select the first column
    else:
        prices = prices.iloc[:, 0]


# Convert prices into a NumPy array
prices = prices.values.reshape(-1, 1)


# Create a scaler
scaler = MinMaxScaler(feature_range=(0, 1))


# Normalize the stock prices
scaled_prices = scaler.fit_transform(prices)


# Create empty lists for input and output
X_stock = []
y_stock = []


# Define the number of previous days
time_steps = 60


# Create sequences from stock prices
for i in range(time_steps, len(scaled_prices)):

    # Store the previous 60 days
    X_stock.append(scaled_prices[i-time_steps:i, 0])

    # Store the next day's price
    y_stock.append(scaled_prices[i, 0])


# Convert lists into NumPy arrays
X_stock = np.array(X_stock)
y_stock = np.array(y_stock)


# Reshape the input for the RNN
X_stock = X_stock.reshape(
    X_stock.shape[0],
    X_stock.shape[1],
    1
)


# Create the stock RNN model
stock_model = Sequential()


# Add an RNN layer with 50 neurons
stock_model.add(
    SimpleRNN(
        50,
        activation="tanh",
        input_shape=(60, 1)
    )
)


# Add an output layer with one neuron
stock_model.add(Dense(1))


# Compile the stock RNN model
stock_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)


# Train the stock RNN model
stock_model.fit(
    X_stock,
    y_stock,
    epochs=5,
    batch_size=32,
    verbose=1
)


# Predict the stock prices
predicted_stock = stock_model.predict(X_stock)


# Convert predictions into original price scale
predicted_stock = scaler.inverse_transform(predicted_stock)


# Convert actual values into original price scale
actual_stock = scaler.inverse_transform(
    y_stock.reshape(-1, 1)
)


# Display the first five predicted prices
print("\nPredicted Google Stock Prices:")
print(predicted_stock[:5])


# Plot actual and predicted prices
plt.figure(figsize=(12, 5))

# Plot actual prices
plt.plot(actual_stock, label="Actual Price")

# Plot predicted prices
plt.plot(predicted_stock, label="Predicted Price")

# Add title
plt.title("Google Stock Price Prediction Using RNN")

# Add x-axis label
plt.xlabel("Trading Days")

# Add y-axis label
plt.ylabel("Stock Price")

# Display legend
plt.legend()

# Show graph
plt.show()

In [ ]:
# SEQUENCE MODELING USING RNN
# ============================================================

print("\n==========================================")
print("SUBQUESTION 2: SEQUENCE MODELING")
print("==========================================")


# Create input sequences
X_sequence = np.array([
    [1, 2, 3],
    [2, 3, 4],
    [3, 4, 5],
    [4, 5, 6],
    [5, 6, 7],
    [6, 7, 8]
], dtype=float)


# Create target values
y_sequence = np.array(
    [4, 5, 6, 7, 8, 9],
    dtype=float
)


# Reshape the input into RNN format
X_sequence = X_sequence.reshape(6, 3, 1)


# Create a sequence RNN model
sequence_model = Sequential()


# Add an RNN layer with 10 neurons
sequence_model.add(
    SimpleRNN(
        10,
        activation="tanh",
        input_shape=(3, 1)
    )
)


# Add an output layer
sequence_model.add(Dense(1))


# Compile the sequence model
sequence_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)


# Train the sequence model
sequence_model.fit(
    X_sequence,
    y_sequence,
    epochs=200,
    verbose=0
)


# Create a new sequence for prediction
test_sequence = np.array(
    [7, 8, 9],
    dtype=float
).reshape(1, 3, 1)


# Predict the next number
sequence_prediction = sequence_model.predict(
    test_sequence,
    verbose=0
)


# Display the predicted number
print(
    "Predicted Next Number:",
    sequence_prediction[0][0]
)

In [ ]:
# CHARACTER RECOGNITION USING RNN AND CNN
# ============================================================

print("\n==========================================")
print("SUBQUESTION 3: CHARACTER RECOGNITION")
print("==========================================")


# Import the MNIST handwritten digit dataset
from tensorflow.keras.datasets import mnist


# Load training and testing data
(X_train, y_train), (X_test, y_test) = mnist.load_data()


# Normalize training images
X_train = X_train.astype("float32") / 255.0


# Normalize testing images
X_test = X_test.astype("float32") / 255.0


# ------------------------------------------------------------
# RNN CHARACTER RECOGNITION
# ------------------------------------------------------------

# Create the RNN model
rnn_model = Sequential()


# Add an RNN layer
rnn_model.add(
    SimpleRNN(
        128,
        activation="tanh",
        input_shape=(28, 28)
    )
)


# Add the output layer for 10 digits
rnn_model.add(
    Dense(10, activation="softmax")
)


# Compile the RNN model
rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# Train the RNN model
rnn_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


# Evaluate the RNN model
rnn_loss, rnn_accuracy = rnn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)


# Display RNN accuracy
print("\nRNN Test Accuracy:", rnn_accuracy)


# ------------------------------------------------------------
# CNN CHARACTER RECOGNITION
# ------------------------------------------------------------

# Reshape images for CNN input
X_train_cnn = X_train.reshape(-1, 28, 28, 1)


# Reshape testing images for CNN
X_test_cnn = X_test.reshape(-1, 28, 28, 1)


# Create the CNN model
cnn_model = Sequential()


# Add the first convolutional layer
cnn_model.add(
    Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    )
)


# Add the first max pooling layer
cnn_model.add(
    MaxPooling2D(pool_size=(2, 2))
)


# Add the second convolutional layer
cnn_model.add(
    Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu"
    )
)


# Add the second max pooling layer
cnn_model.add(
    MaxPooling2D(pool_size=(2, 2))
)


# Flatten the feature maps
cnn_model.add(Flatten())


# Add a fully connected layer
cnn_model.add(
    Dense(128, activation="relu")
)


# Add the output layer
cnn_model.add(
    Dense(10, activation="softmax")
)


# Compile the CNN model
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# Train the CNN model
cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


# Evaluate the CNN model
cnn_loss, cnn_accuracy = cnn_model.evaluate(
    X_test_cnn,
    y_test,
    verbose=0
)


# Display CNN accuracy
print("\nCNN Test Accuracy:", cnn_accuracy)


# ============================================================
# COMPARISON OF RNN AND CNN
# ============================================================

print("\n==========================================")
print("RNN VS CNN COMPARISON")
print("==========================================")


# Display RNN accuracy
print("RNN Accuracy:", rnn_accuracy)


# Display CNN accuracy
print("CNN Accuracy:", cnn_accuracy)


# Create model names
model_names = ["RNN", "CNN"]


# Create accuracy values
accuracy_values = [rnn_accuracy, cnn_accuracy]


# Plot the comparison
plt.figure(figsize=(7, 5))


# Draw the bar chart
plt.bar(model_names, accuracy_values)


# Add the graph title
plt.title("RNN vs CNN Accuracy")


# Add y-axis label
plt.ylabel("Accuracy")


# Show the graph
plt.show()